In [0]:
# Project setup: makes the src/ package importable and loads the config
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.utils.config_loader import load_config
from src.utils.spark_helpers import run_sql_file

cfg = load_config()

In [0]:
%sql
select * from steam_data.default.game_genres

#Analysing how enganging a game is based on genre and Median playtime forever as the metric

In [0]:
%sql
select gg.genre, 
round(mean(g.median_playtime_forever),2) as avg_median_playtime,
approx_percentile(g.median_playtime_forever,0.25) as q1_median_playtime,
median(g.median_playtime_forever) as median_playtime,
approx_percentile(g.median_playtime_forever,0.75) as q3_median_playtime,

count(*) as count_games 
 from steam_data.default.gold_games g join steam_data.default.game_genres gg 
on gg.app_id=g.app_id
where g.median_playtime_forever>0
group by gg.genre
order by median_playtime desc

Databricks visualization. Run in Databricks to view.

**Strategy and RPG:** These are the two most engaging genres by median_playtime_forever. We can see, even the q1 values of strategy and RBP  are much higher than that of other genres. Which means that the less engaging games of theses genres still have a descent engagement as compared to others.
**MMO, Sports and Casual:** These two are highly skewed genres. The avg median playtime for these are very high comapared to other values. This indicates that there are few games which are very popular and drawing maximum engagement. 
**Racing:** Racing has the lowest engagement in terms of time played as expected. This is because racing games are never too long and can be completed in less than an hour ( domain knowledge). This doesnt mean racing games are not popular. People might buy a racing game and finish it without hours . Whereas other games like RPG and Stategy takes days to complete usually.
**Indie/Sim/Adventure/Action:** These games have a skewed distribution as the mean is very high compared to the median , indicating only certain number of games are actually contributing to the numbers. Other than that there isn't much to draw from these. 

Conclusion: Although the genre comparision on basis of median playtime forever gives us some insightm it is still not very helpful in deciding which genre has highest engagement. But we already know it has to be either RPG or Strategy.


The main issue with this analysis is shown by the next cell . Each game might have more than one genre. So, lets say strategy game always are RPG. Then if people love RPG , strategy will also gain engagement and we might endup concluding strategy is as good as RPG which is wrong. 

In [0]:
%sql
SELECT genre_count, COUNT(*) AS games
FROM (SELECT app_id, COUNT(DISTINCT genre) AS genre_count
    FROM steam_data.default.game_genres GROUP BY app_id)
GROUP BY genre_count ORDER BY genre_count

Only around**21000** games has a single genre out of around 130000 games.

##double genres analysis

In [0]:
%sql
SELECT
  a.genre AS genre_1,
  b.genre AS genre_2,
  round(mean(g.median_playtime_forever), 2)           AS avg_median_playtime,
  approx_percentile(g.median_playtime_forever, 0.25)  AS q1_median_playtime,
  median(g.median_playtime_forever)                   AS median_playtime,
  approx_percentile(g.median_playtime_forever, 0.75)  AS q3_median_playtime,
  count(*)                                            AS count_games
FROM steam_data.default.gold_games g
JOIN steam_data.default.game_genres a ON a.app_id = g.app_id
JOIN steam_data.default.game_genres b ON b.app_id = g.app_id
                                     AND a.genre < b.genre   
WHERE g.median_playtime_forever > 0
GROUP BY a.genre, b.genre
HAVING count(*) >= 30                                  
ORDER BY median_playtime DESC

Databricks visualization. Run in Databricks to view.

Among genre pairs, Sports + Strategy has the highest median playtime (540, n = 114) but a very wide spread, and Racing + Strategy (n = 31) is too small to interpret. RPG + Strategy (n ≈ 900) is the most reliable high-engagement pair: its median (368) and Q1 (127) exceed those of either genre alone, and its upper quartile is relatively tight. Strategy appears in 8 of the top 15 pairs, which reinforces it as a strong driver of long playtime.

The heatmap shows that pairs including Strategy have consistently higher median playtime than the partner genre alone, which supports Strategy as a strong candidate for the most engaging genre. However, RPG shows a similar pattern, and the Strategy pairs with the highest medians (Sports, Racing) have small samples. Strategy has a slight edge on median playtime, but this analysis doesn't establish that it is more engaging than RPG.

What if we analyze games with single genre? Does strategy still has the highest engagement. I am doing this analysis to confirm is strategy the key genre. Is strategy alone  doing  good or it always needs a pair to perform well? 

count of games having single genre/per genre

In [0]:
%sql
with onegenre as 
(
SELECT app_id, genre
FROM steam_data.default.game_genres
WHERE app_id IN (
    SELECT app_id
    FROM steam_data.default.game_genres
    GROUP BY app_id
    HAVING COUNT(*) = 1
))
select genre,
round(mean(gg.median_playtime_forever),2) as avg_median_playtime,
approx_percentile(gg.median_playtime_forever,0.25) as q1_median_playtime,
median(gg.median_playtime_forever) as median_metric,
approx_percentile(gg.median_playtime_forever,0.75) as q3_median_playtime,
count(*) as number_of_games
from steam_data.default.gold_games gg join onegenre og on gg.app_id=og.app_id 
where gg.median_playtime_forever>0
group by genre
having number_of_games>50 -- Remove genres with less games
order by median_metric desc

Databricks visualization. Run in Databricks to view.

-> **Filtering out games with 0 median playtime_forever**:
    
-  We can see almost 18000-19000 games get removed after we add the filter of median_playtime_forever>0. This shows many games either are not recorded properly or are not played by many.


->**Finding Median of the median playtime forever**: 

-  In this table the interesting fact is RPG beats strategy . Which means standalone RPG games are better than standalone Strategy. We can ignore MMO as it has only 2 games after all the filtering. 

One possibility from this analysis is that the genre strategy needs to be paired with some other genre/genres. Pure RPG games are still engaging which says pairing them up brings down the metric. 



**Limitation Of This Analysis** 
- A large share of games  have a median playtime of 0 and were excluded. A zero can mean the game was never played, that playtime data is missing, or that at least half of owners never played it even though the rest did. The data can't distinguish these cases, so the results describe games with recorded playtime only and may overstate typical engagement, especially for genres with many obscure or unplayed titles.
- Playtime measures engagement, not popularity, sales, or quality.
- In the overall and pair tables, games with several genres are counted in each genre, so rows overlap.

**CONCLUSION**

**strategy** and **RPG** show the highest typical playtime. Among single-genre games, RPG leads. Results cover games with recorded playtime and measure engagement, not popularity

#SINGLE PLAYER VS MULTIPLAYER

In [0]:
%sql
-- with SM as (
select count(*) from 
    (select app_id ,count(*) as count from steam_data.default.game_categories where category in ('Single-player','Multi-player')
    group by app_id having count>1)
-- )

-- select * from steam_data.default.gold_games gg join SM on gg.app_id=SM.app_id

In [0]:
%sql
SELECT 
    category,
    COUNT(*) AS game_count
FROM steam_data.default.game_categories
WHERE category IN ('Single-player', 'Multi-player')
  AND app_id IN (
      SELECT app_id
      FROM steam_data.default.game_categories
      GROUP BY app_id
      HAVING COUNT(*) = 1
  )
GROUP BY category
ORDER BY game_count DESC;

From the above two tables there are only 5 multiplayer games. Whereas single player and games with both modes say dual mode games are large in number. So the comprision is just between two categories. Single Player vs Dual Mode 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW game_modes AS
SELECT
    app_id,
    CASE
        WHEN COUNT(DISTINCT category) = 1
             AND MAX(category) = 'Single-player'
            THEN 'single'
        WHEN COUNT(DISTINCT category) = 2
            THEN 'dual'
    END AS mode
FROM steam_data.default.game_categories
WHERE category IN ('Single-player', 'Multi-player')
GROUP BY app_id
HAVING COUNT(DISTINCT category) = 1
    OR COUNT(DISTINCT category) = 2;

In [0]:
%sql
select gm.mode,median(median_playtime_forever)  as median_metric  ,count(*) from steam_data.default.gold_games gg join game_modes gm on gg.app_id=gm.app_id 
where mode is not null and median_playtime_forever>0
group by gm.mode order by median_metric desc 